# T09 — ARMA Model (AutoRegressive Moving Average) — v3

**Input:** `data/processed/{train,test}_features.csv`  
**Goal:** predict RUL by forecasting the health-index with ARMA(p,q) until it crosses the failure threshold.

### Why ARMA vs AR
AR only uses past values of the series. ARMA additionally models past **forecast errors** (the MA term), which can reduce lag-response overshoot and give a smoother multi-step forecast. We also enable a linear time trend (`trend='t'`) because the health-index drifts upward — a plain ARMA assumes a stationary mean and would under-forecast degradation.

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
from itertools import product

from src.models.classical import (
    build_pca_health_index,
    compute_failure_threshold,
    predict_rul_arma,
    predict_dataset,
    simulate_test_from_train,
    smooth_series,
    RUL_CAP,
)
from src.evaluation.metrics import evaluate, evaluate_per_subset

PROC_DIR    = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SENSOR_COLS = [f"s{i}" for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]

## 1. Load & prepare

In [ ]:
train = pd.read_csv(PROC_DIR / "train_features.csv")
test  = pd.read_csv(PROC_DIR / "test_features.csv")
print(f"train: {train.shape}  |  test: {test.shape}")

train, test = build_pca_health_index(train, test, SENSOR_COLS)
THRESHOLD = compute_failure_threshold(train, end_of_life_rul=5, quantile=0.5)
print(f"failure threshold: {THRESHOLD:.3f}")

## 2. Demo: ARMA forecast on a single test engine
Shows the smoothed history and the out-of-sample forecast alongside the failure threshold.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA as StatsARIMA

eid = test[test.dataset_id == 1].engine_id.iloc[0]
g   = test[test.engine_id == eid].sort_values("cycle")
raw    = g.health_index.values
smooth = smooth_series(raw, window=5)

model  = StatsARIMA(smooth, order=(2, 0, 2), trend="t").fit()
fcst   = model.forecast(steps=80)

plt.figure(figsize=(10, 4))
plt.plot(range(len(raw)),    raw,    color="lightgray", label="raw")
plt.plot(range(len(smooth)), smooth, color="steelblue", label="smoothed (history)")
plt.plot(range(len(smooth), len(smooth) + len(fcst)), fcst, color="darkorange", label="ARMA forecast")
plt.axhline(THRESHOLD, color="red", ls="--", label=f"threshold={THRESHOLD:.2f}")
plt.xlabel("cycle (from start of test)")
plt.ylabel("health_index")
plt.title(f"ARMA(2,2) forecast — engine {eid}  (true RUL = {int(g.RUL.iloc[-1])})")
plt.legend(); plt.tight_layout(); plt.show()

## 3. (p, q) grid search on a simulated validation set
Random cutoffs on training engines give us labels — we pick the order that minimises simulated RMSE.

In [ ]:
val_df = simulate_test_from_train(train, cutoff_fraction=0.6, max_engines=120, random_seed=42)
print(f"simulated-val engines: {val_df.engine_id.nunique()}")

rows = []
for p, q in product([1, 2, 3], [1, 2, 3]):
    fn = partial(predict_rul_arma, threshold=THRESHOLD, p=p, q=q)
    yt, yp, _ = predict_dataset(val_df, fn)
    m = evaluate(yt, yp, model_name=f"ARMA({p},{q})", verbose=False)
    rows.append({"p": p, "q": q, **m})

tune_df = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
print(tune_df.to_string(index=False))

BEST_P = int(tune_df.iloc[0].p)
BEST_Q = int(tune_df.iloc[0].q)
print(f"\nBest order: ARMA({BEST_P},{BEST_Q})")

## 4. Final evaluation on official test set

In [ ]:
predict_fn = partial(predict_rul_arma, threshold=THRESHOLD, p=BEST_P, q=BEST_Q)
y_true, y_pred, dids = predict_dataset(test, predict_fn)

print(f"=== ARMA({BEST_P},{BEST_Q}) — Test Results ===")
test_metrics = evaluate_per_subset(y_true, y_pred, dids, model_name="ARMA")
display(test_metrics)

## 5. Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(y_true, y_pred, alpha=0.4, s=15, color="steelblue")
axes[0].plot([0, RUL_CAP], [0, RUL_CAP], "r--")
axes[0].set_xlabel("true RUL"); axes[0].set_ylabel("predicted RUL")
axes[0].set_title(f"ARMA({BEST_P},{BEST_Q}) — Pred vs True")

err = y_pred - y_true
axes[1].hist(err, bins=40, color="coral", edgecolor="white")
axes[1].axvline(0, color="black", ls="--")
axes[1].axvline(err.mean(), color="red", label=f"mean={err.mean():+.1f}")
axes[1].set_xlabel("error (pred-true)"); axes[1].set_title("error distribution"); axes[1].legend()

axes[2].hist(y_pred, bins=40, color="mediumpurple", edgecolor="white")
axes[2].set_xlabel("predicted RUL"); axes[2].set_title("prediction distribution")
n_cap = int((y_pred >= RUL_CAP).sum())
print(f"engines predicted at cap: {n_cap}/{len(y_pred)}")
plt.tight_layout(); plt.show()

## 6. Save predictions

In [ ]:
out = pd.DataFrame({
    "model":      "ARMA",
    "p":          BEST_P,
    "q":          BEST_Q,
    "y_true":     y_true,
    "y_pred":     y_pred,
    "dataset_id": dids,
})
out.to_csv(RESULTS_DIR / "T09_ARMA_predictions.csv", index=False)

overall = evaluate(y_true, y_pred, model_name="ARMA_test", verbose=False)
print(f"saved | RMSE={overall['rmse']:.4f}  NASA={overall['nasa_score']:.2f}")